## Vanilla Mamba — Inference Notebook

Load the trained Mamba checkpoint and generate text from prompts.

## 1 · Imports & Config

In [1]:
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer
from model import MambaModel

# ── Model hyper-parameters (must match training) ────────────────────
VOCAB_SIZE = 4096
D_MODEL    = 256
N_LAYERS   = 6
D_STATE    = 16
D_CONV     = 4
EXPAND     = 2

# ── Special tokens ─────────────────────────────────────────────────
BOS_ID = 2
EOS_ID = 3

# ── Paths ──────────────────────────────────────────────────────────
CHECKPOINT_PATH = "mamba1_best.pt"
TOKENIZER_PATH  = "../../tokenizer_4k.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


## 2 · Load Tokenizer

In [2]:
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
print(f"Tokenizer loaded  ·  vocab size = {tokenizer.get_vocab_size()}")

Tokenizer loaded  ·  vocab size = 4096


## 3 · Load Model & Checkpoint

In [3]:
model = MambaModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    d_state=D_STATE,
    d_conv=D_CONV,
    expand=EXPAND,
).to(DEVICE)

state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded  ·  {n_params / 1e6:.2f}M parameters")

Model loaded  ·  4.63M parameters


## 4 · Generation Utilities

In [4]:
@torch.no_grad()
def generate(
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.95,
):
    """
    Auto-regressive generation with temperature, top-k, and nucleus (top-p) sampling.
    """
    encoded = tokenizer.encode(prompt)
    input_ids = torch.tensor(
        [BOS_ID] + encoded.ids, dtype=torch.long
    ).unsqueeze(0).to(DEVICE)

    for _ in range(max_new_tokens):
        logits = model(input_ids)
        next_logits = logits[:, -1, :] / temperature

        # ── Top-k filtering ────────────────────────────────────────
        if top_k > 0:
            top_k_vals, _ = torch.topk(next_logits, top_k)
            threshold = top_k_vals[:, -1].unsqueeze(-1)
            next_logits = next_logits.masked_fill(next_logits < threshold, -float('inf'))

        # ── Nucleus (top-p) filtering ──────────────────────────────
        if top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            remove_mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[remove_mask] = -float('inf')
            next_logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)

        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        if next_token.item() == EOS_ID:
            break

    return tokenizer.decode(input_ids[0].tolist())


def show(prompt, **kwargs):
    """Pretty-print a generation."""
    text = generate(prompt, **kwargs)
    print(f"\n{'─' * 60}")
    print(f"Prompt │ {prompt}")
    print(f"{'─' * 60}")
    print(text)
    print(f"{'─' * 60}\n")

## 5 · Generate Stories 📖

In [5]:
show("Once upon a time")


────────────────────────────────────────────────────────────
Prompt │ Once upon a time
────────────────────────────────────────────────────────────
Once upon a time , there was a little girl named Lily . She loved to play outside and explore . One day , she was playing in the snow when she heard a loud noise . She went over to her mommy and said , " Mommy , can I play with the slide ? It ' s so cold and loud . I want to play with you !" mommy said , " Let me go . Come on , Lily , let me ride my bike . I will hold my hand and you will help me . I will find her ." Lily was happy . She rode her bike down the slide . She played on the slide , and the slide . When they got to the bottom , Lily saw a bird with a hurt wing . She picked it up and said , " Mommy , I don ' t have a ball !" Her mom smiled and said , " Yes , you can try . But be careful , it ' s so hard to keep up ." Lily was happy to see her doll and thought it was so cool . She wanted
───────────────────────────────────────────

In [6]:
show("The little cat")


────────────────────────────────────────────────────────────
Prompt │ The little cat
────────────────────────────────────────────────────────────
The little cat was so excited that he started playing . He had a fun day . The end .
────────────────────────────────────────────────────────────



In [7]:
show("There was a boy named")


────────────────────────────────────────────────────────────
Prompt │ There was a boy named
────────────────────────────────────────────────────────────
There was a boy named Timmy . Timmy loved to play with his favorite toy , a ball , and a doll . One day , Timmy ' s mom gave him a spoon . He had a big bowl of chocolate cake . Timmy was so happy to have it . He wore it everywhere he went , and it made him very happy . He felt happy and healthy . He was very creative and always looked after his mom .
────────────────────────────────────────────────────────────



## 6 · Experiment with Sampling Parameters

Try different temperatures, top-k, and top-p values to see how generation changes.

In [8]:
prompt = "Once upon a time"

print("=" * 60)
print("GREEDY  (temperature → 0)")
print("=" * 60)
show(prompt, temperature=0.1, top_k=1)

print("=" * 60)
print("WARM  (temperature = 0.8, top_k = 50)")
print("=" * 60)
show(prompt, temperature=0.8, top_k=50)

print("=" * 60)
print("CREATIVE  (temperature = 1.2, top_p = 0.9)")
print("=" * 60)
show(prompt, temperature=1.2, top_k=0, top_p=0.9)

GREEDY  (temperature → 0)

────────────────────────────────────────────────────────────
Prompt │ Once upon a time
────────────────────────────────────────────────────────────
Once upon a time , there was a little girl named Lily . She loved to play outside in the park with her friends . One day , Lily ' s friend came to visit her . Lily saw her friend ' s house and asked her what was wrong . Lily told her friend that she had lost her favorite toy . Her friend was very sad and didn ' t know what to do . Lily ' s friend said , " Don ' t worry , Lily . I will help you ." Lily was happy and said , " Thank you , Lily . You are my friend ." Lily forgave her brother and they continued to play together . They had a great time playing together .
────────────────────────────────────────────────────────────

WARM  (temperature = 0.8, top_k = 50)

────────────────────────────────────────────────────────────
Prompt │ Once upon a time
────────────────────────────────────────────────────────────
Once